In [ ]:
# Cell 1 — Load config
%run /home/jovyan/work/setup/config.py
import sys; sys.path.insert(0, "/home/jovyan/work")
from utils.delta_utils import save_layer

In [ ]:
# Cell 2 — Build dim_date from Silver date range
from pyspark.sql.functions import col, date_format, month, year, quarter, dayofweek, expr
from pyspark.sql.types import IntegerType

df_silver = spark.read.format("delta").load(f"{SILVER_PATH}/silver_beverage_sales_enriched")

min_date, max_date = df_silver.selectExpr("MIN(full_date)", "MAX(full_date)").first()
print(f"Date range: {min_date} to {max_date}")

date_spine = spark.sql(f"""
    SELECT EXPLODE(SEQUENCE(DATE '{min_date}', DATE '{max_date}', INTERVAL 1 DAY)) AS full_date
""")

dim_date = (
    date_spine
    .withColumn("date_sk",    expr("CAST(DATE_FORMAT(full_date, 'yyyyMMdd') AS INT)"))
    .withColumn("year",       year(col("full_date")))
    .withColumn("month",      month(col("full_date")))
    .withColumn("month_name", date_format(col("full_date"), "MMMM"))
    .withColumn("quarter",    expr("CAST(CEIL(MONTH(full_date) / 3.0) AS INT)"))
    .withColumn("period",     month(col("full_date")))
    .withColumn("day_of_week",date_format(col("full_date"), "EEEE"))
    .withColumn("is_weekend", expr("DAYOFWEEK(full_date) IN (1, 7)"))
    .select("date_sk", "full_date", "year", "month", "month_name",
            "quarter", "period", "day_of_week", "is_weekend")
)

save_layer(dim_date, "dim_date", GOLD_PATH, PG_WRITE_PROPS)
dim_date.show(5)